In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import kagglehub
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder


df = pd.read_csv(f"{path}/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=10, edgecolor='black')
plt.title('delievery time distribution')
plt.xlabel('delievry time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:

df=df.drop("Order_ID", axis=1)

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

df = df.dropna(subset=['Weather', 'Traffic_Level','Time_of_Day','Courier_Experience_yrs','Delivery_Time'])

def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Task 3: Write your code here:

def check_duplicates(df_clean):
  duplicates = df_clean.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)



In [ ]:
# Task 4: Write your code here:

categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df

In [ ]:
# Task 5: Write your code here:

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, "Delivery_Time")


In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.head())

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
# Show full dataset class distribution
full_ratio = (y.value_counts(normalize=True) * 100).sort_index()
print("Full Dataset Class Distribution")
print("  y class percentages:", {k: f"{v:.2f}%" for k, v in full_ratio.items()})
print("-" * 40)

# Define Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified K-Fold Cross Validation\n" + "-"*40)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)

    # showing class distribution
    train_ratio = (y_train.value_counts(normalize=True) * 100).sort_index()
    test_ratio = (y_test.value_counts(normalize=True) * 100).sort_index()

    print("  y_train class percentages:", {k: f"{v:.2f}%" for k, v in train_ratio.items()})
    print("  y_test class percentages :", {k: f"{v:.2f}%" for k, v in test_ratio.items()})

    model.fit()

    print("-" * 40)

In [ ]:
# Task 1: Write your code here:

feature_importance = pd.DataFrame({
    'feature': X,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here: